This PySpark code retrieves yesterday’s date in MMddyyyy format and uses it to dynamically construct a table name, setting specific Spark configurations for date handling in Parquet files.

In [1]:
%%pyspark
from pyspark.sql.functions import date_format, current_date, concat, lit

# Set configurations
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY")
spark.conf.set("spark.sql.parquet.int96RebaseModeInWrite", "LEGACY")

# Get the current date in the desired format
current_date_format = spark.sql("SELECT date_format(current_date()-1, 'MMddyyyy') AS current_date_format").collect()[0]['current_date_format']

# Construct the table name dynamically
table_name = f"Snapshot_Turns_{current_date_format}"

print("Table Name: ", table_name)

StatementMeta(, c2f178b0-7113-477f-a27c-10ba06ebcbcf, 3, Finished, Available, Finished)

Table Name:  Snapshot_Turns_09292024


This PySpark code creates a table with specified columns and data types using the Delta format, only if the table does not already exist.

In [16]:
%%pyspark
# Create the table if it doesn't exist
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {table_name} (
    Item_Number STRING,
    Location_Code STRING,
    Unit_Cost DOUBLE,
    Total_R12_Qty DOUBLE,
    Total_R6_Qty DOUBLE,
    Inventory DOUBLE,
    CSC STRING,
    Loc_Type STRING,
    Inventory_Cost DOUBLE,
    Parts_Sales_Cost_R12 DOUBLE
)
USING delta
""")


StatementMeta(, 84172afd-4a77-4e65-81c7-b8cc2fed8c2d, 18, Finished, Available, Finished)

DataFrame[]

This PySpark SQL inserts query into a specified table using the "table_name" variable initialized above.

In [4]:
%%pyspark
spark.sql(f"""
INSERT INTO {table_name}
-- Union of the four queries (12 month)
WITH CombinedResults12 AS (
    -- invoiced work order parts rolling 12
    SELECT
        WOL.No_ AS Item_Number,
        WOL.Location_Code,
        I.Unit_Cost,
        SUM(SIL.Quantity) AS R12_Qty
    FROM DE_LH_SILVER.Work_Order_Header AS WOH
    JOIN DE_LH_SILVER.Work_Order_Line AS WOL
        ON WOH.No_ = WOL.Document_No_
    JOIN DE_LH_SILVER.Sales_Invoice_Line AS SIL
        ON WOL.Document_No_ = SIL.Document_No_
        AND WOL.Line_No_ = SIL.Work_Order_Line_No_
    JOIN DE_LH_SILVER.Location AS LOC 
        ON WOL.Service_Van = LOC.Code
    JOIN DE_LH_SILVER.Item AS I
    	ON WOL.No_ = I.No_ 
    WHERE 
        WOL.Type = 1
        AND WOH.Posting_Status = 2
        AND WOH.Posting_Date >= date_add(current_date(), -365)
        AND WOH.Posting_Date <= current_date()
        AND SIL.Quantity <> 0
        AND SIL.Quantity IS NOT NULL
    GROUP BY 
        WOL.No_,
        WOL.Location_Code,
        I.Unit_Cost 
        
    UNION ALL
    
    -- parts work orders not invoiced to customer rolling 12
    SELECT
        WOL.No_ AS Item_Number,
        WOL.Location_Code,
        I.Unit_Cost,
        SUM(WOL.Quantity) AS R12_Qty
    FROM DE_LH_SILVER.Work_Order_Header AS WOH
    JOIN DE_LH_SILVER.Work_Order_Line AS WOL
        ON WOH.No_ = WOL.Document_No_
    LEFT JOIN DE_LH_SILVER.Sales_Invoice_Line AS SIL
        ON WOL.Document_No_ = SIL.Document_No_
        AND WOL.Line_No_ = SIL.Work_Order_Line_No_
    JOIN DE_LH_SILVER.Location AS LOC 
        ON WOL.Service_Van = LOC.Code
    JOIN DE_LH_SILVER.Item AS I
    	ON WOL.No_ = I.No_  
    WHERE 
        WOL.Type = 1
        AND WOH.Posting_Status = 2
        AND WOH.Posting_Date >= date_add(current_date(), -365)
        AND WOH.Posting_Date <= current_date()
        AND SIL.Document_No_ IS NULL
    GROUP BY 
        WOL.No_,
        WOL.Location_Code,
        I.Unit_Cost
    HAVING 
        SUM(WOL.Quantity) <> 0
        
    UNION ALL
    
    -- parts sales invoiced no work order
    SELECT
        SIL.No_ AS Item_Number,
        SIL.Location_Code,
        I.Unit_Cost,
        SUM(SIL.Quantity) AS R12_Qty
    FROM DE_LH_SILVER.Sales_Invoice_Line AS SIL
    LEFT JOIN 
        DE_LH_SILVER.Work_Order_Header AS WOH
        ON SIL.Document_No_ = WOH.No_
    JOIN DE_LH_SILVER.Item AS I
    	ON SIL.No_ = I.No_ 
    WHERE SIL.Gen_Prod_Posting_Group = 'PARTS'
        AND SIL.Posting_Date >= date_add(current_date(), -365)
        AND SIL.Posting_Date <= current_date()
        AND WOH.No_ IS NULL
        AND SIL.Quantity IS NOT NULL
        AND SIL.Type = 2
        AND SIL.Quantity <> 0
    GROUP BY 
        SIL.No_,
        SIL.Location_Code,
        I.Unit_Cost
        
    UNION ALL
    
    -- parts sales credit memo (returns)
	SELECT
		SCML.No_ AS Item_Number,
		SCML.Location_Code,
		I.Unit_Cost,
		(SUM(SCML.Quantity * -1)) AS R12_Qty
	FROM DE_LH_SILVER.Sales_Credit_Memo_Header AS SCMH
	JOIN DE_LH_SILVER.Sales_Credit_Memo_Line AS SCML
		ON SCMH.No_ = SCML.Document_No_
	JOIN DE_LH_SILVER.Location AS LOC
		ON SCML.Location_Code = LOC.Code
	JOIN DE_LH_SILVER.Item AS I
    	ON SCML.No_ = I.No_ 
	WHERE SCML.Gen_Prod_Posting_Group = 'PARTS'
		AND SCML.Type = 2
		AND SCMH.Posting_Date >= date_add(current_date(), -365)
	    AND SCMH.Posting_Date <= current_date()
	GROUP BY
		SCML.No_,
		SCML.Location_Code,
		I.Unit_Cost
),

-- Union of the four queries (6 month)
CombinedResults6 AS (
    -- invoiced work order parts rolling 6
    SELECT
        WOL.No_ AS Item_Number,
        WOL.Location_Code,
        I.Unit_Cost,
        SUM(SIL.Quantity) AS R6_Qty
    FROM DE_LH_SILVER.Work_Order_Header AS WOH
    JOIN DE_LH_SILVER.Work_Order_Line AS WOL
        ON WOH.No_ = WOL.Document_No_
    JOIN DE_LH_SILVER.Sales_Invoice_Line AS SIL
        ON WOL.Document_No_ = SIL.Document_No_
        AND WOL.Line_No_ = SIL.Work_Order_Line_No_
    JOIN DE_LH_SILVER.Location AS LOC 
        ON WOL.Service_Van = LOC.Code
    JOIN DE_LH_SILVER.Item AS I
    	ON WOL.No_ = I.No_ 
    WHERE 
        WOL.Type = 1
        AND WOH.Posting_Status = 2
        AND WOH.Posting_Date >= date_add(current_date(), -183)
        AND WOH.Posting_Date <= current_date()
        AND SIL.Quantity <> 0
        AND SIL.Quantity IS NOT NULL
    GROUP BY 
        WOL.No_,
        WOL.Location_Code,
        I.Unit_Cost
        
    UNION ALL
    
    -- parts work orders not invoiced to customer rolling 6
    SELECT
        WOL.No_ AS Item_Number,
        WOL.Location_Code,
        I.Unit_Cost,
        SUM(WOL.Quantity) AS R6_Qty
    FROM DE_LH_SILVER.Work_Order_Header AS WOH
    JOIN DE_LH_SILVER.Work_Order_Line AS WOL
        ON WOH.No_ = WOL.Document_No_
    LEFT JOIN DE_LH_SILVER.Sales_Invoice_Line AS SIL
        ON WOL.Document_No_ = SIL.Document_No_
        AND WOL.Line_No_ = SIL.Work_Order_Line_No_
    JOIN DE_LH_SILVER.Location AS LOC 
        ON WOL.Service_Van = LOC.Code
    JOIN DE_LH_SILVER.Item AS I
    	ON WOL.No_ = I.No_ 
    WHERE 
        WOL.Type = 1
        AND WOH.Posting_Status = 2
        AND WOH.Posting_Date >= date_add(current_date(), -183)
        AND WOH.Posting_Date <= current_date()
        AND SIL.Document_No_ IS NULL
    GROUP BY 
        WOL.No_,
        WOL.Location_Code,
        I.Unit_Cost
    HAVING 
        SUM(WOL.Quantity) <> 0
        
    UNION ALL
    
    -- parts sales invoiced no work order
    SELECT
        SIL.No_ AS Item_Number,
        SIL.Location_Code,
        I.Unit_Cost,
        SUM(SIL.Quantity) AS R6_Qty
    FROM DE_LH_SILVER.Sales_Invoice_Line AS SIL
    LEFT JOIN 
        DE_LH_SILVER.Work_Order_Header AS WOH
        ON SIL.Document_No_ = WOH.No_
    JOIN DE_LH_SILVER.Item AS I
    	ON SIL.No_ = I.No_ 
    WHERE SIL.Gen_Prod_Posting_Group = 'PARTS'
        AND SIL.Posting_Date >= date_add(current_date(), -183)
        AND SIL.Posting_Date <= current_date()
        AND WOH.No_ IS NULL
        AND SIL.Quantity IS NOT NULL
        AND SIL.Type = 2
        AND SIL.Quantity <> 0
    GROUP BY 
        SIL.No_,
        SIL.Location_Code,
        I.Unit_Cost
        
    UNION ALL
    
    -- parts sales credit memo (returns)
	SELECT
		SCML.No_ AS Item_Number,
		SCML.Location_Code,
		I.Unit_Cost,
		(SUM(SCML.Quantity * -1)) AS R6_Qty
	FROM DE_LH_SILVER.Sales_Credit_Memo_Header AS SCMH
	JOIN DE_LH_SILVER.Sales_Credit_Memo_Line AS SCML
		ON SCMH.No_ = SCML.Document_No_
	JOIN DE_LH_SILVER.Location AS LOC
		ON SCML.Location_Code = LOC.Code
	JOIN DE_LH_SILVER.Item AS I
    	ON SCML.No_ = I.No_ 
	WHERE SCML.Gen_Prod_Posting_Group = 'PARTS'
		AND SCML.Type = 2
		AND SCMH.Posting_Date >= date_add(current_date(), -183)
	    AND SCMH.Posting_Date <= current_date()
	GROUP BY
		SCML.No_,
		SCML.Location_Code,
		I.Unit_Cost
),

-- Location and responsibility center details
LocationDetails AS (
    SELECT 
        DE_LH_SILVER.Location.Code AS Location_Code,
        DE_LH_SILVER.Location.Responsibility_Center AS CSC,
        CASE 
            Location_Type
            WHEN 1 THEN 'Central'
            ELSE 'Van'
        END AS Loc_Type
    FROM 
        DE_LH_SILVER.Location
),

-- Inventory details
InventoryDetails AS (
    SELECT 
        ile.Item_No_ AS Item_Number,
        ile.Location_Code,
        SUM(ile.Remaining_Quantity) AS Inventory
    FROM 
        DE_LH_BRONZE_ELC.Item_Ledger_Entry AS ile 
    WHERE 
        ile.Open_ = 1
        AND ile.Remaining_Quantity > 0
    GROUP BY 
        ile.Location_Code,
        ile.Item_No_
),
-- PreCost_Aggregation
PreCost_Aggregation AS (
    SELECT 
        CR12.Item_Number,
        CR12.Location_Code,
        CR12.Unit_Cost,
        COALESCE(CR12.Total_R12_Qty, 0) AS Total_R12_Qty,
        COALESCE(CR6.Total_R6_Qty, 0) AS Total_R6_Qty,
        COALESCE(ID.Inventory, 0) AS Inventory,
        LD.CSC,
        LD.Loc_Type
    FROM 
        (SELECT 
            Item_Number, 
            Location_Code,
            Unit_Cost,
            SUM(R12_Qty) AS Total_R12_Qty
        FROM CombinedResults12
        GROUP BY Item_Number, Location_Code, Unit_Cost
        ) AS CR12
    LEFT JOIN 
        (SELECT 
            Item_Number, 
            Location_Code, 
            SUM(R6_Qty) AS Total_R6_Qty
        FROM CombinedResults6
        GROUP BY Item_Number, Location_Code
        ) AS CR6
    ON 
        CR12.Item_Number = CR6.Item_Number
        AND CR12.Location_Code = CR6.Location_Code
    LEFT JOIN 
        InventoryDetails AS ID
    ON 
        CR12.Item_Number = ID.Item_Number
        AND CR12.Location_Code = ID.Location_Code
    LEFT JOIN 
        LocationDetails AS LD
    ON 
        CR12.Location_Code = LD.Location_Code
    ORDER BY 
        CR12.Item_Number,
        CR12.Location_Code
)
    SELECT
        *,
        PAg.Inventory * PAg.Unit_Cost AS Inventory_Cost,
        PAg.Total_R12_Qty * PAg.Unit_Cost AS Parts_Sales_Cost_R12
    FROM PreCost_Aggregation AS PAg;
""")

StatementMeta(, , , Waiting, , Waiting)

DataFrame[]

View a sample of the newly created table.

In [8]:
%%pyspark

# Get the current date in the desired format
current_date_format = spark.sql("SELECT date_format(current_date()-1, 'MMddyyyy') AS current_date_format").collect()[0]['current_date_format']

# Construct the table name dynamically
table_name = f"Snapshot_Turns_{current_date_format}"

# Create the query string with the table name
query = f"SELECT * FROM {table_name}"

# Execute the query
spark.sql(query).show()


StatementMeta(, 105a2458-e989-4fd0-a658-5f71614dbb77, 10, Finished, Available, Finished)

+-----------------+-------------+---------+-------------+------------+---------+---+--------+--------------+--------------------+
|      Item_Number|Location_Code|Unit_Cost|Total_R12_Qty|Total_R6_Qty|Inventory|CSC|Loc_Type|Inventory_Cost|Parts_Sales_Cost_R12|
+-----------------+-------------+---------+-------------+------------+---------+---+--------+--------------+--------------------+
|          DC18210|          815|      1.5|          2.0|         2.0|      6.0| 10|     Van|           9.0|                 3.0|
|          DC56540|          815|      1.5|          2.0|         2.0|      6.0| 10|     Van|           9.0|                 3.0|
|       DCMFDR3X10|          870|     4.95|          2.0|         2.0|      6.0| 10|     Van|          29.7|                 9.9|
|    DDGDH11GA#8HI|          760|     3.47|          2.0|         2.0|      6.0| 10|     Van|         20.82|                6.94|
|        FT1034366|          647|     4.63|          2.0|         2.0|      6.0|  9|     V

Append the snapshot table to the master turns snapshot table with a new column.

In [2]:
%%pyspark
# Load the Delta table into a DataFrame
df = spark.read.table(table_name)
df_with_filename = df.withColumn("File_Name", lit(table_name))

# Show the first few rows of the DataFrame
df_with_filename.show()

StatementMeta(, c2f178b0-7113-477f-a27c-10ba06ebcbcf, 4, Finished, Available, Finished)

+------------+-------------+---------+-------------+------------+---------+---+--------+--------------+--------------------+--------------------+
| Item_Number|Location_Code|Unit_Cost|Total_R12_Qty|Total_R6_Qty|Inventory|CSC|Loc_Type|Inventory_Cost|Parts_Sales_Cost_R12|           File_Name|
+------------+-------------+---------+-------------+------------+---------+---+--------+--------------+--------------------+--------------------+
|BTSF10100BLK|          688|    5.617|          1.0|         0.0|     13.0|  3|     Van|        73.021|               5.617|Snapshot_Turns_09...|
|BTSF10100RED|          609|     4.46|          1.0|         0.0|     11.0|  3|     Van|         49.06|                4.46|Snapshot_Turns_09...|
|  COCPB00047|            6|    0.368|          1.0|         0.0|     39.0|  6| Central|        14.352|               0.368|Snapshot_Turns_09...|
|  DCDOTH2425|          881|      3.1|          1.0|         0.0|     14.0| 10|     Van|          43.4|                 3.1|

In [19]:
%%pyspark

df_with_filename.write.format("delta").mode("append").saveAsTable("Turns_Snapshots")

StatementMeta(, 84172afd-4a77-4e65-81c7-b8cc2fed8c2d, 21, Finished, Available, Finished)